In [1]:
"""
Step 2 of 2: read the CSVs built by build_q1_timeseries.py and run the
actual analysis. No raw file reading here -- this only touches the small,
already-summarized daily/weekly CSVs, so it's fast to re-run with different
settings (lag range, which columns to compare, etc).

WHAT "lag-0 correlation" MEANS:
  Compare day t's burnout count against day t's CVE count, directly,
  using both Pearson (linear, sensitive to outlier days) and Spearman
  (rank-based, robust to outliers). One number summarizing same-day overlap.

WHAT "lag scan" MEANS:
  Repeat that same comparison, but shift the CVE column backward/forward
  in time relative to the burnout column, at every offset from -MAX_LAG_DAYS
  to +MAX_LAG_DAYS. Positive lag = CVE count from N days BEFORE the burnout
  measurement (tests "does a crisis lead to burnout later"). Negative lag
  is a sanity check in the other direction. The output is a full curve --
  correlation strength at every offset -- not a single number. Read off
  where that curve peaks to find the delay (if any) with the strongest match.
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q1_analysis/"
DAILY_CSV = os.path.join(OUT_DIR, "q1_daily_series.csv")

BAT_COLS = ["n_bat_ge1", "n_bat_ge2"]
CVE_COLS = ["n_cve_run1", "n_cve_run2"]

MAX_LAG_DAYS = 84
LAG_STEP_DAYS = 7


def lag_scan(series_a, series_b, max_lag, step):
    """Pearson r between series_a(t) and series_b(t - lag) at every lag step."""
    results = []
    for lag in range(-max_lag, max_lag + 1, step):
        shifted = series_b.shift(lag)
        valid = series_a.notna() & shifted.notna()
        if valid.sum() < 10:
            continue
        r, p = stats.pearsonr(series_a[valid], shifted[valid])
        results.append({"lag_days": lag, "pearson_r": r, "p_value": p, "n": int(valid.sum())})
    return pd.DataFrame(results)


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}")
        print("Run build_q1_timeseries.py first to generate it.")
        return

    daily = pd.read_csv(DAILY_CSV)
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    # ------------------------------------------------------------------
    # Lag-0 correlations, all bat_col x cve_col combinations
    # ------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("LAG-0 CORRELATIONS")
    print("=" * 70)
    rows = []
    for bat_col in BAT_COLS:
        for cve_col in CVE_COLS:
            r_p, p_p = stats.pearsonr(daily[bat_col], daily[cve_col])
            r_s, p_s = stats.spearmanr(daily[bat_col], daily[cve_col])
            rows.append({
                "bat_threshold": bat_col, "cve_threshold": cve_col,
                "pearson_r": round(r_p, 4), "pearson_p": round(p_p, 4),
                "spearman_r": round(r_s, 4), "spearman_p": round(p_s, 4),
            })
    corr_df = pd.DataFrame(rows)
    print(corr_df.to_string(index=False))
    corr_df.to_csv(os.path.join(OUT_DIR, "q1_correlations_lag0.csv"), index=False)

    # ------------------------------------------------------------------
    # Lag scan, all combinations
    # ------------------------------------------------------------------
    print("\n" + "=" * 70)
    print(f"LAG SCAN (-{MAX_LAG_DAYS} to +{MAX_LAG_DAYS} days, step {LAG_STEP_DAYS})")
    print("Positive lag = CVE count from N days BEFORE the burnout measurement")
    print("=" * 70)

    scans = {}
    fig, axes = plt.subplots(1, len(CVE_COLS), figsize=(7 * len(CVE_COLS), 5), sharey=True)
    if len(CVE_COLS) == 1:
        axes = [axes]
    colors = {"n_bat_ge1": "tab:blue", "n_bat_ge2": "tab:purple"}

    for ax_idx, cve_col in enumerate(CVE_COLS):
        ax = axes[ax_idx]
        for bat_col in BAT_COLS:
            key = f"{bat_col}_vs_{cve_col}"
            scan = lag_scan(daily[bat_col], daily[cve_col], MAX_LAG_DAYS, LAG_STEP_DAYS)
            scans[key] = scan
            scan.to_csv(os.path.join(OUT_DIR, f"q1_lag_scan_{key}.csv"), index=False)

            best = scan.loc[scan["pearson_r"].abs().idxmax()]
            print(f"\n{key}: peak |r| at lag={int(best['lag_days'])} days "
                  f"(r={best['pearson_r']:.4f}, p={best['p_value']:.4g})")

            ax.plot(scan["lag_days"], scan["pearson_r"], marker="o",
                    label=bat_col, color=colors.get(bat_col))

        ax.axvline(0, color="gray", linestyle="--", linewidth=1)
        ax.axhline(0, color="gray", linestyle="--", linewidth=1)
        ax.set_title(cve_col)
        ax.set_xlabel("Lag (days) -- positive = CVE leads burnout")
        if ax_idx == 0:
            ax.set_ylabel("Pearson r")
        ax.legend()

    fig.suptitle("Lag scan: bat_score thresholds vs. CVE severity thresholds")
    fig.tight_layout()
    plot_path = os.path.join(OUT_DIR, "q1_lag_scan_plot.png")
    fig.savefig(plot_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"\nSaved plot -> {plot_path}")

    print("\nDone. Compare the four peak lags and r-values above -- same lag across all")
    print("four = a real timing pattern; r increasing toward the strictest thresholds")
    print("(bat>=2, run2) = severity tracks severity.")


if __name__ == "__main__":
    main()

Loaded 3071 days from /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_daily_series.csv

LAG-0 CORRELATIONS
bat_threshold cve_threshold  pearson_r  pearson_p  spearman_r  spearman_p
    n_bat_ge1    n_cve_run1     0.1599        0.0      0.3102         0.0
    n_bat_ge1    n_cve_run2     0.1503        0.0      0.1799         0.0
    n_bat_ge2    n_cve_run1     0.0872        0.0      0.2049         0.0
    n_bat_ge2    n_cve_run2     0.1201        0.0      0.1499         0.0

LAG SCAN (-84 to +84 days, step 7)
Positive lag = CVE count from N days BEFORE the burnout measurement

n_bat_ge1_vs_n_cve_run1: peak |r| at lag=42 days (r=0.1831, p=3.103e-24)

n_bat_ge2_vs_n_cve_run1: peak |r| at lag=-42 days (r=0.1089, p=1.846e-09)

n_bat_ge1_vs_n_cve_run2: peak |r| at lag=-42 days (r=0.1526, p=3.091e-17)

n_bat_ge2_vs_n_cve_run2: peak |r| at lag=21 days (r=0.1247, p=4.723e-12)

Saved plot -> /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_lag_scan_plot.png

Done. Compare the four peak lags 